In [3]:
!pip install pymupdf

   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
    --------------------------------------- 0.3/19.8 MB ? eta -:--:--
   - -------------------------------------- 0.8/19.8 MB 2.4 MB/s eta 0:00:08
   --- ------------------------------------ 1.6/19.8 MB 3.0 MB/s eta 0:00:07
   ----- ---------------------------------- 2.6/19.8 MB 3.6 MB/s eta 0:00:05
   ------- -------------------------------- 3.7/19.8 MB 4.0 MB/s eta 0:00:05
   ---------- ----------------------------- 5.2/19.8 MB 4.5 MB/s eta 0:00:04
   ------------- -------------------------- 6.6/19.8 MB 4.9 MB/s eta 0:00:03
   ----------------- ---------------------- 8.7/19.8 MB 5.4 MB/s eta 0:00:03
   --------------------- ------------------ 10.7/19.8 MB 6.0 MB/s eta 0:00:02
   ------------------------- -------------- 12.8/19.8 MB 6.4 MB/s eta 0:00:02
   ------------------------------ --------- 15.2/19.8 MB 6.8 MB/s eta 0:00:01
   ---------------------------------- ----- 17.3/19.8 MB 7.1 MB/s eta 0:00:01
   ------


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pymupdf

In [7]:
pdf_path = r"c:\Users\Asus\Downloads\HDFC_OptimaSecure_Policy_Doc.pdf"

doc = pymupdf.open(pdf_path)
doc

Document('c:\Users\Asus\Downloads\HDFC_OptimaSecure_Policy_Doc.pdf')

In [9]:
from collections import Counter


doc = pymupdf.open(pdf_path)

font_sizes = []

for page in doc:

    page_dict = page.get_text("dict")

    for block in page_dict["blocks"]:

        if "lines" not in block:
            continue

        for line in block["lines"]:

            for span in line["spans"]:

                text = span["text"].strip()

                if text:
                    font_sizes.append(round(span["size"], 1))


font_counts = Counter(font_sizes)

print(font_counts)

body_font_size = font_counts.most_common(1)[0][0]

print("Detected body font size:", body_font_size)

Counter({15.9: 143, 12.0: 102, 9.9: 50, 20.9: 37, 10.9: 33, 13.4: 33, 18.0: 31, 8.5: 31, 14.0: 23, 11.0: 20, 16.0: 16, 28.0: 16, 11.9: 16, 18.9: 10, 13.9: 7, 27.0: 6, 36.2: 6, 19.9: 6, 4.3: 5, 13.5: 5, 5.8: 5, 6.7: 4, 8.0: 4, 42.7: 3, 12.2: 3, 9.0: 3, 7.0: 3, 20.0: 2, 9.3: 2, 4.9: 2, 25.0: 2, 16.4: 1, 21.0: 1, 56.3: 1, 21.1: 1, 5.2: 1, 7.9: 1, 4.4: 1, 49.9: 1, 10.6: 1, 15.0: 1, 8.2: 1, 7.8: 1, 11.6: 1, 14.1: 1, 7.6: 1})
Detected body font size: 15.9


In [11]:

import re
# -------------------------------------------------------
# Find most common font size
# -------------------------------------------------------

font_sizes = []

for page in doc:

    page_dict = page.get_text("dict")

    for block in page_dict["blocks"]:

        if "lines" not in block:
            continue

        for line in block["lines"]:

            for span in line["spans"]:

                text = span["text"].strip()

                if text:
                    font_sizes.append(
                        round(span["size"], 1)
                    )


font_counts = Counter(font_sizes)

body_font_size = font_counts.most_common(1)[0][0]

print("Body font size:", body_font_size)


# -------------------------------------------------------
# Heading detector
# -------------------------------------------------------

def is_heading(text, font_size, font_name):

    text = text.strip()

    if not text:
        return False

    is_bold = "bold" in font_name.lower()

    is_larger = font_size > body_font_size + 1

    numbered = bool(
        re.match(
            r"^\d+(\.\d+)*\.?\s+\S+",
            text
        )
    )

    uppercase = (
        text.isupper()
        and len(text.split()) <= 12
    )

    short_text = len(text.split()) <= 20

    return short_text and (
        is_larger
        or (is_bold and numbered)
        or uppercase
    )


# -------------------------------------------------------
# Check whether bbox is inside table
# -------------------------------------------------------

def bbox_inside_table(bbox, table_boxes):

    x0, y0, x1, y1 = bbox

    center_x = (x0 + x1) / 2
    center_y = (y0 + y1) / 2

    for tx0, ty0, tx1, ty1 in table_boxes:

        if (
            tx0 <= center_x <= tx1
            and
            ty0 <= center_y <= ty1
        ):
            return True

    return False


# -------------------------------------------------------
# Parse document
# -------------------------------------------------------

elements = []

for page_number, page in enumerate(doc, start=1):

    # ---------------------------------------------------
    # Detect tables first
    # ---------------------------------------------------

    table_finder = page.find_tables()

    table_boxes = []

    for table in table_finder.tables:

        table_boxes.append(table.bbox)

        elements.append({
            "type": "table",
            "page": page_number,
            "bbox": table.bbox,
            "data": table.extract()
        })


    # ---------------------------------------------------
    # Extract text blocks
    # ---------------------------------------------------

    page_dict = page.get_text("dict")

    for block in page_dict["blocks"]:

        if "lines" not in block:
            continue

        block_text = []
        sizes = []
        fonts = []

        for line in block["lines"]:

            for span in line["spans"]:

                text = span["text"].strip()

                if not text:
                    continue

                block_text.append(text)

                sizes.append(span["size"])

                fonts.append(span["font"])


        if not block_text:
            continue


        text = " ".join(block_text)

        bbox = block["bbox"]


        # Ignore text contained inside table
        if bbox_inside_table(
            bbox,
            table_boxes
        ):
            continue


        # Use dominant / largest formatting
        max_font_size = max(sizes)

        font_name = fonts[
            sizes.index(max_font_size)
        ]


        # Classify element
        if is_heading(
            text,
            max_font_size,
            font_name
        ):

            content_type = "heading"

        else:

            content_type = "paragraph"


        elements.append({
            "type": content_type,
            "page": page_number,
            "text": text,
            "bbox": bbox,
            "font_size": max_font_size,
            "font": font_name
        })

Body font size: 15.9


In [12]:
for element in elements:

    print("\nTYPE:", element["type"])
    print("PAGE:", element["page"])

    if element["type"] == "table":

        for row in element["data"]:
            print(row)

    else:

        print(element["text"])


TYPE: table
PAGE: 1
['', '', '', '', '', '', '', '', '']
[None, None, None, None, '', None, None, None, None]
[None, None, '', None, '', None, None, None, None]
[None, None, None, None, '', None, None, None, None]

TYPE: heading
PAGE: 1
A HEALTH COVER THAT STAYS ONE STEP AHEAD

TYPE: paragraph
PAGE: 2
So Much More...

TYPE: heading
PAGE: 2
Optima Secure is a testament to the trust that 1.54+ crore @ customers have bestowed upon us.

TYPE: paragraph
PAGE: 2
It’s great when you ask for something and get more in return, isn’t it? That’s why, HDFC ERGO brings to you a health insurance plan that gives you SO MUCH more than you asked for!

TYPE: paragraph
PAGE: 2
HDFC ERGO Optima Secure provides 4X coverage* , at no additional cost, thereby redeﬁning the value you get from health insurance. It doesn’t just secure your present, but safeguards your future as well.

TYPE: heading
PAGE: 2
Introduction

TYPE: heading
PAGE: 3
Key Coverages

TYPE: paragraph
PAGE: 3
Secure Beneﬁt is a ﬁrst-of-its-k